# Emotion Music Studio — Colab GPU (built-in proxy, no DNS/cloudflared)

Serves the app through Colab's own `googleusercontent.com` proxy, so there's **no
cloudflared tunnel and no DNS issue** (no `NXDOMAIN`).

**Run on Google Colab, not locally.** First: **Runtime → Change runtime type → T4 GPU**,
then **Runtime → Run all**. Open the URL printed by the last cell.

Open in Colab:
`https://colab.research.google.com/github/AthSri0507/Multi_Modal-Music-Generation/blob/main/deploy/colab_gpu_proxy.ipynb`

In [ ]:
# 0. Guard: this notebook is for Google Colab (Linux GPU), not a local machine.
import os, sys
try:
    import google.colab  # noqa: F401
except ImportError:
    raise SystemExit('Open this in Google Colab (colab.research.google.com), not locally.')
print('On Colab — good to go.')

In [ ]:
# 1. Fresh clone of the LATEST code (avoids reusing a stale clone).
%cd /content
!rm -rf repo
!git clone --depth 1 https://github.com/AthSri0507/Multi_Modal-Music-Generation.git repo
%cd repo
!git log -1 --oneline

In [ ]:
# 2. Install serving deps. Colab already ships CUDA torch — do NOT reinstall it.
!pip install -q transformers sentencepiece protobuf soundfile fastapi uvicorn python-multipart librosa
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime to T4 GPU')

In [ ]:
# 3. Build the React UI FIRST (the server only serves it if dist exists at startup).
import shutil, os
if shutil.which('node'):
    !cd frontend && npm install --silent && npm run build
else:
    print('node not found (unusual on Colab) — will run API-only.')
print('UI built:', os.path.exists('frontend/dist/index.html'))

In [ ]:
# 4. Launch the backend (AFTER the UI build) and wait until it's healthy.
#    Default model = small (fast, no big download). For higher quality, change to:
#    MODEL = 'facebook/musicgen-medium'   (downloads ~3.5GB on the first generate)
import subprocess, time, urllib.request
MODEL = 'facebook/musicgen-small'
!pkill -f uvicorn
env = {**os.environ, 'PYTHONPATH': os.getcwd(), 'MUSICGEN_MODEL_NAME': MODEL, 'MUSICGEN_USE_CLAP': '1'}
log = open('server.log', 'w')
subprocess.Popen([sys.executable, '-m', 'uvicorn', 'src.api.app:app', '--host', '0.0.0.0', '--port', '8000'],
                 cwd=os.getcwd(), env=env, stdout=log, stderr=subprocess.STDOUT)
ok = False
for _ in range(60):
    try:
        print('UP:', urllib.request.urlopen('http://localhost:8000/api/v1/health', timeout=3).read().decode()); ok = True; break
    except Exception:
        time.sleep(2)
if not ok:
    print('backend did not start — error log:\n', open('server.log').read()[-3000:])

In [ ]:
# 5. Warm up the model once (so the first click in the UI isn't slow / doesn't stall).
import urllib.request, json
print('warming up', MODEL, '...')
req = urllib.request.Request('http://localhost:8000/api/v1/music/generate',
    data=json.dumps({'prompt': 'warm up', 'duration': 3, 'preset': 'fast'}).encode(),
    headers={'Content-Type': 'application/json'})
print('ready, sample id:', json.loads(urllib.request.urlopen(req, timeout=1200).read())['id'])

In [ ]:
# 6. Open the app — Colab's built-in proxy (googleusercontent.com, no DNS/cloudflared).
from google.colab.output import eval_js
print('Open your studio here:')
print(eval_js("google.colab.kernel.proxyPort(8000)"))

**Notes**
- The printed `googleusercontent.com` URL opens in the same browser you're signed into Colab with.
- If the page is blank/JSON: the UI wasn't built — re-run cell 3 (build), then cell 4 (relaunch), then cell 6.
- Higher quality: set `MODEL = 'facebook/musicgen-medium'` in cell 4 and re-run cells 4→6 (first generate downloads ~3.5GB once).
- The gallery (SQLite) + audio reset when the Colab runtime stops; mount Google Drive to persist.